# LangChain Hands-on 17 — LangGraph Basics (engine behind agents)
**Trainer:** Ajeetkumar

**What we will practise**
- Why LangGraph?
- State, Nodes, Edges
- Simple graph → conditional routing
- Chatbot graph with messages
- Tool-calling graph (build your own agent)
- Memory with checkpointer
- Human-in-the-loop with interrupt

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

In [ ]:
! pip install -q langchain langchain-groq langgraph

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('groq_key')

In [ ]:
import os
os.environ['GROQ_API_KEY'] = groq_api_key

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b")

# Why LangGraph?
- `create_agent` is built on **LangGraph**.
- Use LangGraph directly when you need **custom flows**: loops, branches, approvals, multi-agent.

| Word | Meaning |
|---|---|
| **State** | shared data (a dict) that flows through the graph |
| **Node** | a Python function that reads state and returns updates |
| **Edge** | which node runs next |
| **Conditional edge** | choose the next node with an if / else function |
| **START / END** | entry and exit |

## 1. First graph — no LLM, just understand the flow

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
class State(TypedDict):
    name: str
    greeting: str

In [ ]:
def say_hello(state: State):
    return {"greeting": f"Hello {state['name']}!"}

def add_emoji(state: State):
    return {"greeting": state["greeting"] + " 👋"}

In [ ]:
builder = StateGraph(State)
builder.add_node("say_hello", say_hello)
builder.add_node("add_emoji", add_emoji)
builder.add_edge(START, "say_hello")
builder.add_edge("say_hello", "add_emoji")
builder.add_edge("add_emoji", END)
graph = builder.compile()

In [ ]:
graph.invoke({"name": "Ajeet"})

In [ ]:
from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())

## 2. Conditional edge — route by marks

In [ ]:
class Result(TypedDict):
    marks: int
    message: str

def check(state: Result):
    return state

def passed(state: Result):
    return {"message": "🎉 Pass"}

def failed(state: Result):
    return {"message": "📚 Fail, try again"}

def route(state: Result):
    return "passed" if state["marks"] >= 35 else "failed"

In [ ]:
b = StateGraph(Result)
b.add_node("check", check); b.add_node("passed", passed); b.add_node("failed", failed)
b.add_edge(START, "check")
b.add_conditional_edges("check", route, {"passed": "passed", "failed": "failed"})
b.add_edge("passed", END); b.add_edge("failed", END)
result_graph = b.compile()

In [ ]:
result_graph.invoke({"marks": 72}), result_graph.invoke({"marks": 20})

## 3. Chatbot graph (LLM node + messages state)

In [ ]:
from langgraph.graph import MessagesState

In [ ]:
def chatbot(state: MessagesState):
    return {"messages": [llm.invoke(state["messages"])]}

b = StateGraph(MessagesState)
b.add_node("chatbot", chatbot)
b.add_edge(START, "chatbot")
b.add_edge("chatbot", END)
chat_graph = b.compile()

In [ ]:
out = chat_graph.invoke({"messages": [{"role": "user", "content": "Hi! Explain LangGraph in one line."}]})
out["messages"][-1].content

## 4. Build your own agent: LLM + tools + loop

In [ ]:
from langchain.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

@tool
def get_weather(city: str) -> str:
    """Get weather of a city."""
    return f"{city}: 29°C, sunny"

tools = [multiply, get_weather]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
def assistant(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

In [ ]:
b = StateGraph(MessagesState)
b.add_node("assistant", assistant)
b.add_node("tools", ToolNode(tools))
b.add_edge(START, "assistant")
b.add_conditional_edges("assistant", tools_condition)      # tool call? → "tools" else → END
b.add_edge("tools", "assistant")                           # loop back
agent_graph = b.compile()

In [ ]:
Image(agent_graph.get_graph().draw_mermaid_png())

In [ ]:
out = agent_graph.invoke({"messages": [{"role": "user", "content": "What is 23 * 17, and weather in Pune?"}]})
for m in out["messages"]:
    m.pretty_print()

## 5. Memory with a checkpointer

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
memory_graph = b.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "1"}}

In [ ]:
memory_graph.invoke({"messages": [{"role": "user", "content": "My name is Ajeet. Multiply 6 and 7."}]}, config)["messages"][-1].content

In [ ]:
memory_graph.invoke({"messages": [{"role": "user", "content": "What is my name? Multiply the last answer by 2."}]}, config)["messages"][-1].content

In [ ]:
memory_graph.get_state(config).values["messages"][-1].content

## 6. Human-in-the-loop — pause before tools

In [ ]:
hitl_graph = b.compile(checkpointer=InMemorySaver(), interrupt_before=["tools"])
cfg = {"configurable": {"thread_id": "approve-1"}}

In [ ]:
hitl_graph.invoke({"messages": [{"role": "user", "content": "Multiply 12 and 12"}]}, cfg)
hitl_graph.get_state(cfg).next          # ('tools',) → paused before tools

In [ ]:
hitl_graph.get_state(cfg).values["messages"][-1].tool_calls

In [ ]:
# human says OK → continue by passing None
hitl_graph.invoke(None, cfg)["messages"][-1].content

# LangChain vs LangGraph
| Need | Use |
|---|---|
| prompt → llm → parser | LCEL chain |
| standard tool-calling agent | `create_agent` |
| custom steps, loops, branches, approvals, multi-agent | **LangGraph** |

Next to learn (LangChain Academy – Intro to LangGraph): reducers, sub-graphs, multi-agent supervisor, long-term memory store, streaming, deployment.

## 🧪 Practice
1. Build a graph: `classify` (complaint / query) → `complaint_node` or `query_node`.
2. Add a `convert_currency` tool to the agent graph.
3. Use `interrupt_before` to approve a `send_email` tool.